# PPO Optimizer-Chain Training Notebook


In [ ]:
from pathlib import Path
import os
import sys
import warnings

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mplconfig"))

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from gymnasium import spaces
from sklearn.model_selection import train_test_split
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.monitor import Monitor

RANDOM_STATE = 42
RUNTIME_WEIGHT = 0.1
INVALID_STOP_PENALTY = -1.0
TOTAL_TIMESTEPS = 30_000
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "dbs" / "original_data" / "one_and_two_opt_master.csv").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not locate project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

CHAIN_OPTIMIZER_DIR = PROJECT_ROOT / "machine_learning" / "RL" / "chain_optimizer"
if str(CHAIN_OPTIMIZER_DIR) not in sys.path:
    sys.path.insert(0, str(CHAIN_OPTIMIZER_DIR))

from optimizer_chain_env import METRIC_NAMES, OptimizerChainEnv

MODEL_PATH = CHAIN_OPTIMIZER_DIR / f"optimizer_chain_ppo_{RUNTIME_WEIGHT}.zip"

np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)


## 1. Load and Prepare Data

In [ ]:
CSV_PATH = PROJECT_ROOT / "dbs" / "original_data" / "one_and_two_opt_master.csv"

master_df = pd.read_csv(CSV_PATH)

required_columns = [
    "original_circuit",
    "original_category",
    "chain_name",
    "initial_num_qubits",
    "initial_depth",
    "initial_two_qubit_gates",
    "initial_total_gates",
    "initial_two_qubit_depth",
    "initial_gate_density",
    "initial_two_qubit_ratio",
    "final_num_qubits",
    "final_depth",
    "final_two_qubit_gates",
    "final_total_gates",
    "final_two_qubit_depth",
    "total_runtime_seconds",
]

missing = sorted(set(required_columns) - set(master_df.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = master_df.copy()
df["two_qubit_gate_reduction"] = (
    df["initial_two_qubit_gates"] - df["final_two_qubit_gates"]
)
df["two_qubit_gate_reduction_pct"] = (
    100
    * df["two_qubit_gate_reduction"]
    / df["initial_two_qubit_gates"].replace(0, np.nan)
)

df = df.dropna(
    subset=[
        "two_qubit_gate_reduction_pct",
        "total_runtime_seconds",
        "final_two_qubit_gates",
    ]
).copy()

OPTIMIZER_NAMES = [
    "qiskit_standard",
    "qiskit_ai",
    "tket",
    "wisq_rules",
    "wisq_bqskit",
]
STOP_ACTION = len(OPTIMIZER_NAMES)
CATEGORY_NAMES = sorted(df["original_category"].unique())

summary = pd.DataFrame(
    {
        "rows": [len(df)],
        "original_circuits": [df["original_circuit"].nunique()],
        "chains": [df["chain_name"].nunique()],
        "categories": [df["original_category"].nunique()],
        "error_true_rows": [
            int(df.get("error", pd.Series(False, index=df.index)).astype(bool).sum())
        ],
    }
)
summary


In [ ]:
category_summary = (
    df.groupby("original_category")
    .agg(rows=("original_circuit", "size"), circuits=("original_circuit", "nunique"))
    .sort_values("circuits", ascending=False)
)
category_summary


## 2. Circuit-Level Train / Validation / Test Split

The split is by `original_circuit`, not by row, so all optimizer-chain results for a circuit stay in the same split.

In [ ]:
circuits = np.array(sorted(df["original_circuit"].unique()))

train_val_circuits, test_circuits = train_test_split(
    circuits,
    test_size=0.10,
    random_state=RANDOM_STATE,
)
train_circuits, val_circuits = train_test_split(
    train_val_circuits,
    test_size=0.10 / 0.90,
    random_state=RANDOM_STATE,
)

train_df = df[df["original_circuit"].isin(train_circuits)].copy()
val_df = df[df["original_circuit"].isin(val_circuits)].copy()
test_df = df[df["original_circuit"].isin(test_circuits)].copy()

assert set(train_circuits).isdisjoint(val_circuits)
assert set(train_circuits).isdisjoint(test_circuits)
assert set(val_circuits).isdisjoint(test_circuits)

split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(train_df), len(val_df), len(test_df)],
        "circuits": [
            train_df["original_circuit"].nunique(),
            val_df["original_circuit"].nunique(),
            test_df["original_circuit"].nunique(),
        ],
        "chains": [
            train_df["chain_name"].nunique(),
            val_df["chain_name"].nunique(),
            test_df["chain_name"].nunique(),
        ],
    }
)
split_summary


## 3. Reward Definition

The reward is utility: `utility = percent_reduction_score - 0.20 * normalized_log_runtime`. Runtime is normalized with the 99th percentile of training runtimes so extreme outliers do not dominate the reward.

In [ ]:
train_runtime_99 = float(train_df["total_runtime_seconds"].quantile(0.99))
runtime_denominator = np.log1p(max(train_runtime_99, 1e-9))


def add_reward_columns(frame):
    frame = frame.copy()
    frame["pct_score"] = frame["two_qubit_gate_reduction_pct"] / 100.0
    frame["runtime_score"] = (
        np.log1p(frame["total_runtime_seconds"].clip(lower=0))
        / runtime_denominator
    ).clip(0, 2)
    frame["utility_reward"] = (
        frame["pct_score"] - RUNTIME_WEIGHT * frame["runtime_score"]
    )
    return frame


train_df = add_reward_columns(train_df)
val_df = add_reward_columns(val_df)
test_df = add_reward_columns(test_df)
df = add_reward_columns(df)

reward_summary = pd.DataFrame(
    {
        "train_runtime_99_seconds": [train_runtime_99],
        "runtime_weight": [RUNTIME_WEIGHT],
        "train_utility_mean": [train_df["utility_reward"].mean()],
        "train_utility_min": [train_df["utility_reward"].min()],
        "train_utility_max": [train_df["utility_reward"].max()],
    }
)
reward_summary


In [ ]:
train_df["runtime_score"].describe()

In [ ]:
chain_reward_summary = (
    train_df.groupby("chain_name")
    .agg(
        avg_utility=("utility_reward", "mean"),
        avg_pct_reduction=("two_qubit_gate_reduction_pct", "mean"),
        avg_runtime=("total_runtime_seconds", "mean"),
    )
    .sort_values("avg_utility", ascending=False)
)
chain_reward_summary.head(10)


## 4. Observation Scaling

In [ ]:
def initial_metric_frame(frame):
    return pd.DataFrame(
        {
            "num_qubits": frame["initial_num_qubits"],
            "depth": frame["initial_depth"],
            "two_qubit_gates": frame["initial_two_qubit_gates"],
            "total_gates": frame["initial_total_gates"],
            "two_qubit_depth": frame["initial_two_qubit_depth"],
            "gate_density": frame["initial_gate_density"],
            "two_qubit_ratio": frame["initial_two_qubit_ratio"],
        }
    )


def final_metric_frame(frame):
    final_total = frame["final_total_gates"].replace(0, np.nan)
    final_qubits = frame["final_num_qubits"].replace(0, np.nan)
    return pd.DataFrame(
        {
            "num_qubits": frame["final_num_qubits"],
            "depth": frame["final_depth"],
            "two_qubit_gates": frame["final_two_qubit_gates"],
            "total_gates": frame["final_total_gates"],
            "two_qubit_depth": frame["final_two_qubit_depth"],
            "gate_density": frame["final_total_gates"] / final_qubits,
            "two_qubit_ratio": frame["final_two_qubit_gates"] / final_total,
        }
    ).fillna(0.0)


train_single_df = train_df[~train_df["chain_name"].str.contains("__", regex=False)]
scale_source = pd.concat(
    [initial_metric_frame(train_df), final_metric_frame(train_single_df)],
    ignore_index=True,
)
metric_scale = scale_source.quantile(0.95).replace(0, 1.0).to_dict()
metric_scale


## 5. Gymnasium Environment

The environment class is imported from `optimizer_chain_env.py` so training and evaluation use the same environment definition.

In [ ]:
train_env_raw = OptimizerChainEnv(
    train_df,
    OPTIMIZER_NAMES,
    CATEGORY_NAMES,
    metric_scale,
    shuffle_each_epoch=True,
)
check_env(train_env_raw, warn=True)
train_env = Monitor(train_env_raw)

val_env = OptimizerChainEnv(
    val_df,
    OPTIMIZER_NAMES,
    CATEGORY_NAMES,
    metric_scale,
    shuffle_each_epoch=False,
)

test_env = OptimizerChainEnv(
    test_df,
    OPTIMIZER_NAMES,
    CATEGORY_NAMES,
    metric_scale,
    shuffle_each_epoch=False,
)


## 6. Hyperparameter Tuning Workspace

Edit `PPO_CONFIG` and `TOTAL_TIMESTEPS` above or below, rerun training, and use the validation set to compare candidate settings. No automated tuning is run in this notebook yet.

In [ ]:
PPO_CONFIG = {
    "policy": "MlpPolicy",
    "learning_rate": 0.001,
    "n_steps": 256,
    "batch_size": 64,
    "gamma": 1.0,
    "ent_coef": 0.05,
    "verbose": 0,
    "seed": RANDOM_STATE,
}

PPO_CONFIG


In [ ]:
def evaluate_saved_policy_on_frame(model, frame, method="PPO agent"):
    env = OptimizerChainEnv(
        frame,
        OPTIMIZER_NAMES,
        CATEGORY_NAMES,
        metric_scale,
        shuffle_each_epoch=False,
    )
    selected = []

    for circuit in sorted(frame["original_circuit"].unique()):
        obs, _ = env.reset(options={"circuit": circuit})
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(int(action))

        if not (terminated or truncated):
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(int(action))

        selected_chain = info["selected_chain"]
        row = frame[
            (frame["original_circuit"] == circuit)
            & (frame["chain_name"] == selected_chain)
        ].iloc[0].copy()
        row["method"] = method
        row["selected_chain"] = selected_chain
        selected.append(row)

    return pd.DataFrame(selected)


# After training, use this cell as a quick validation check while tuning hyperparameters.
# Example:
# val_selected = evaluate_saved_policy_on_frame(model, val_df)
# val_selected[["original_circuit", "selected_chain", "utility_reward"]].head()


### Hyper Parameter Tuning Grid Search



In [ ]:
from itertools import product

RUN_GRID_SEARCH = True
TUNING_TIMESTEPS = 10_000

search_grid = {
    "learning_rate": [1e-4, 3e-4, 1e-3],
    "n_steps": [256, 512, 1024],
    "batch_size": [64, 128],
    "ent_coef": [0.0, 0.01, 0.05],
    "gamma": [1.0],
}

if RUN_GRID_SEARCH:
    grid_results = []
    total_configs = np.prod([len(values) for values in search_grid.values()])

    for config_idx, (learning_rate, n_steps, batch_size, ent_coef, gamma) in enumerate(
        product(
            search_grid["learning_rate"],
            search_grid["n_steps"],
            search_grid["batch_size"],
            search_grid["ent_coef"],
            search_grid["gamma"],
        ),
        start=1,
    ):
        print(
            f"[{config_idx}/{total_configs}] "
            f"lr={learning_rate}, n_steps={n_steps}, "
            f"batch_size={batch_size}, ent_coef={ent_coef}, gamma={gamma}"
        )

        candidate_train_env = Monitor(
            OptimizerChainEnv(
                train_df,
                OPTIMIZER_NAMES,
                CATEGORY_NAMES,
                metric_scale,
                shuffle_each_epoch=True,
            )
        )

        candidate_model = PPO(
            PPO_CONFIG["policy"],
            candidate_train_env,
            learning_rate=learning_rate,
            n_steps=n_steps,
            batch_size=batch_size,
            gamma=gamma,
            ent_coef=ent_coef,
            verbose=0,
            seed=RANDOM_STATE,
        )
        candidate_model.learn(total_timesteps=TUNING_TIMESTEPS)

        val_selected = evaluate_saved_policy_on_frame(
            candidate_model,
            val_df,
            method="PPO validation",
        )

        grid_results.append(
            {
                "learning_rate": learning_rate,
                "n_steps": n_steps,
                "batch_size": batch_size,
                "ent_coef": ent_coef,
                "gamma": gamma,
                "avg_val_utility": val_selected["utility_reward"].mean(),
                "avg_val_runtime_seconds": val_selected["total_runtime_seconds"].mean(),
                "avg_val_reduction_pct": val_selected[
                    "two_qubit_gate_reduction_pct"
                ].mean(),
                "selected_unique_chains": val_selected["selected_chain"].nunique(),
            }
        )

    grid_results_df = (
        pd.DataFrame(grid_results)
        .sort_values("avg_val_utility", ascending=False)
        .reset_index(drop=True)
    )
else:
    grid_results_df = pd.DataFrame()
    print("Set RUN_GRID_SEARCH = True to run hyperparameter tuning.")

grid_results_df


## 7. Train PPO Agent

In [ ]:
class RewardHistoryCallback(BaseCallback):
    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.episode_rewards = []

    def _on_step(self) -> bool:
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self.episode_rewards.append(info["episode"]["r"])
        return True


if "grid_results_df" in globals() and not grid_results_df.empty:
    best_grid_row = grid_results_df.iloc[0]
    FINAL_PPO_CONFIG = PPO_CONFIG.copy()
    FINAL_PPO_CONFIG.update(
        {
            "learning_rate": float(best_grid_row["learning_rate"]),
            "n_steps": int(best_grid_row["n_steps"]),
            "batch_size": int(best_grid_row["batch_size"]),
            "gamma": float(best_grid_row["gamma"]),
            "ent_coef": float(best_grid_row["ent_coef"]),
        }
    )
    print("Using best hyperparameters from grid_results_df:")
else:
    FINAL_PPO_CONFIG = PPO_CONFIG.copy()
    print("grid_results_df not found or empty; using PPO_CONFIG:")

print(FINAL_PPO_CONFIG)

model = PPO(
    FINAL_PPO_CONFIG["policy"],
    train_env,
    learning_rate=FINAL_PPO_CONFIG["learning_rate"],
    n_steps=FINAL_PPO_CONFIG["n_steps"],
    batch_size=FINAL_PPO_CONFIG["batch_size"],
    gamma=FINAL_PPO_CONFIG["gamma"],
    ent_coef=FINAL_PPO_CONFIG["ent_coef"],
    verbose=FINAL_PPO_CONFIG["verbose"],
    seed=FINAL_PPO_CONFIG["seed"],
)

reward_callback = RewardHistoryCallback()
model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=reward_callback)
print(f"Completed PPO training for {TOTAL_TIMESTEPS:,} timesteps.")


In [ ]:
rewards = np.array(reward_callback.episode_rewards, dtype=float)

plt.figure(figsize=(10, 4))
if len(rewards) > 0:
    window = max(20, min(200, len(rewards) // 10))
    smoothed = pd.Series(rewards).rolling(window=window, min_periods=1).mean()
    plt.plot(rewards, alpha=0.25, label="episode reward")
    plt.plot(smoothed, linewidth=2, label=f"rolling mean ({window})")
plt.xlabel("Episode")
plt.ylabel("Utility reward")
plt.title("Training Reward History")
plt.legend()
plt.tight_layout()
plt.show()


## 8. Save Model

In [ ]:
model.save(MODEL_PATH)
print(f"Saved model to {MODEL_PATH.resolve()}")
